In [1]:
# Uncomment only if this kernel is missing the required packages.
# Recommended for macOS Apple Silicon / PyTorch MPS.
# %pip install -U pip
# %pip install torch torchvision torchaudio
# %pip install segmentation-models-pytorch timm albumentations opencv-python-headless pandas
# Restart the kernel after installing PyTorch or segmentation-models-pytorch.


# Advanced Baseline: Local Segmentation Training + Threshold Tuning + TTA + Pseudo Labels

Readable upgrade over `baselane.ipynb` for local runs on macOS Apple Silicon.


In [2]:
from __future__ import annotations

import json
import random
import time
from contextlib import nullcontext
from copy import deepcopy
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn


# =========================
# CONFIG
# =========================
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
MASK_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}


def guess_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'dl-lab-3-product-segmentation').exists():
            return candidate
    return cwd


PROJECT_ROOT = guess_project_root()
DATA_ROOT = PROJECT_ROOT / 'dl-lab-3-product-segmentation'

LABELED_IMAGES_DIR = DATA_ROOT / 'train' / 'images'
LABELED_MASKS_DIR = DATA_ROOT / 'train' / 'masks'
UNLABELED_DIR = DATA_ROOT / 'unlabeled' / 'images'
EXTERNAL_UNLABELED_DIR = PROJECT_ROOT / 'external_unlabeled' / 'images'  # optional
TEST_IMAGES_DIR = DATA_ROOT / 'test_images'
CLASSIFICATION_RECURSIVE_DIR = PROJECT_ROOT / 'dl-lab-1-image-classification'
SAVE_DIR = PROJECT_ROOT / 'seg_runs' / 'advanced_baseline'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SPECS = [
    {
        'name': 'unetpp_resnet34',
        'model_name': 'UnetPlusPlus',
        'encoder_name': 'resnet34',
        'encoder_weights': 'imagenet',
    },
    {
        'name': 'fpn_resnet34',
        'model_name': 'FPN',
        'encoder_name': 'resnet34',
        'encoder_weights': 'imagenet',
    },
    {
        'name': 'unet_resnet34',
        'model_name': 'Unet',
        'encoder_name': 'resnet34',
        'encoder_weights': 'imagenet',
    },
]
DEFAULT_MODEL_SPEC = MODEL_SPECS[0]
NUM_CLASSES = 1
ACTIVATION = None

IMG_SIZE = 384
BATCH_SIZE = 8
NUM_EPOCHS = 60
LR = 3e-4
WEIGHT_DECAY = 1e-4
N_FOLDS = 3
NUM_WORKERS = 0
SEED = 42
USE_AMP = False
SMOKE_TEST_ONE_EPOCH = False
EARLY_STOPPING_PATIENCE = 10
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 2
MIN_LR = 1e-6
DEFAULT_THRESHOLD = 0.50
THRESHOLD_GRID = [round(x, 2) for x in np.arange(0.30, 0.71, 0.05)]

RESUME_FROM_LAST = True
GROUP_SPLIT_BY_CAMERA = True
USE_EMA = True
EMA_DECAY = 0.999

ENSEMBLE_PIXEL_SAMPLES = 120_000
ENSEMBLE_MIN_WEIGHT = 0.05
RUN_TEST_INFERENCE = False
RUN_PSEUDO_LABELING = True
RUN_CLASSIFICATION_RECURSIVE_PSEUDO = True
SAVE_PROBABILITY_MAPS = True
POSTPROCESS_MIN_COMPONENT_AREA_RATIO = 0.0005
POSTPROCESS_CLOSE_KERNEL = 5

PSEUDO_SAVE_ONLY_CONFIDENT = False
PSEUDO_MIN_MEAN_CONFIDENCE = 0.80
PSEUDO_MIN_PIXEL_CONFIDENCE = None
PSEUDO_MIN_FOREGROUND_PROB = 0.75
PSEUDO_MIN_AREA_RATIO = 0.002
PSEUDO_MAX_AREA_RATIO = 0.85

# Night-run notes:
# - BATCH_SIZE = 8 is a safe default for Apple Silicon.
# - If memory is stable, you can try BATCH_SIZE = 10 later.
# - If you want more quality, you can try IMG_SIZE = 416 later.
# - Do not change both at once without a smoke test.


def get_device() -> torch.device:
    mps_backend = getattr(torch.backends, 'mps', None)
    if mps_backend is not None and mps_backend.is_available():
        return torch.device('mps')
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


DEVICE = get_device()
PIN_MEMORY = DEVICE.type == 'cuda'


# =========================
# UTILS
# =========================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def read_image(path: Path, flags: int = cv2.IMREAD_COLOR):
    data = np.fromfile(str(path), dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, flags)


def save_png_mask(path: Path, mask: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, encoded = cv2.imencode('.png', mask)
    if not ok:
        raise RuntimeError(f'Failed to encode mask for {path}')
    encoded.tofile(str(path))


def save_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')


def load_json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding='utf-8'))


def count_files(path: Path, extensions: set[str]) -> int:
    if not path.exists():
        return 0
    return sum(1 for file_path in path.rglob('*') if file_path.is_file() and file_path.suffix.lower() in extensions)


def print_path_status(label: str, path: Path, extensions: set[str], required: bool) -> None:
    if path.exists():
        print(f'[OK] {label}: {path} | files={count_files(path, extensions)}')
    elif required:
        print(f'[MISSING] {label}: {path}')
    else:
        print(f'[OPTIONAL] {label}: {path} | missing -> will be skipped')


def build_config_payload(best_threshold: float | None = None) -> dict:
    return {
        'project_root': str(PROJECT_ROOT),
        'paths': {
            'labeled_images_dir': str(LABELED_IMAGES_DIR),
            'labeled_masks_dir': str(LABELED_MASKS_DIR),
            'unlabeled_dir': str(UNLABELED_DIR),
            'external_unlabeled_dir': str(EXTERNAL_UNLABELED_DIR),
            'test_images_dir': str(TEST_IMAGES_DIR),
            'classification_recursive_dir': str(CLASSIFICATION_RECURSIVE_DIR),
            'save_dir': str(SAVE_DIR),
        },
        'model_specs': MODEL_SPECS,
        'train': {
            'img_size': IMG_SIZE,
            'batch_size': BATCH_SIZE,
            'num_epochs': NUM_EPOCHS,
            'lr': LR,
            'weight_decay': WEIGHT_DECAY,
            'num_workers': NUM_WORKERS,
            'seed': SEED,
            'use_amp': USE_AMP,
            'smoke_test_one_epoch': SMOKE_TEST_ONE_EPOCH,
            'early_stopping_patience': EARLY_STOPPING_PATIENCE,
            'scheduler_factor': SCHEDULER_FACTOR,
            'scheduler_patience': SCHEDULER_PATIENCE,
            'min_lr': MIN_LR,
            'default_threshold': DEFAULT_THRESHOLD,
            'threshold_grid': THRESHOLD_GRID,
            'resume_from_last': RESUME_FROM_LAST,
            'use_ema': USE_EMA,
            'ema_decay': EMA_DECAY,
        },
        'folds': {
            'n_folds': N_FOLDS,
            'group_split_by_camera': GROUP_SPLIT_BY_CAMERA,
            'group_key': 'camera_ip_prefix',
        },
        'ensemble': {
            'pixel_samples': ENSEMBLE_PIXEL_SAMPLES,
            'min_weight': ENSEMBLE_MIN_WEIGHT,
        },
        'runtime': {
            'device': DEVICE.type,
            'pin_memory': PIN_MEMORY,
        },
        'postprocess': {
            'min_component_area_ratio': POSTPROCESS_MIN_COMPONENT_AREA_RATIO,
            'close_kernel': POSTPROCESS_CLOSE_KERNEL,
        },
        'pseudo_labeling': {
            'save_only_confident': PSEUDO_SAVE_ONLY_CONFIDENT,
            'min_mean_confidence': PSEUDO_MIN_MEAN_CONFIDENCE,
            'min_pixel_confidence': PSEUDO_MIN_PIXEL_CONFIDENCE,
            'min_foreground_prob': PSEUDO_MIN_FOREGROUND_PROB,
            'min_area_ratio': PSEUDO_MIN_AREA_RATIO,
            'max_area_ratio': PSEUDO_MAX_AREA_RATIO,
            'run_classification_recursive_pseudo': RUN_CLASSIFICATION_RECURSIVE_PSEUDO,
        },
        'best_threshold': best_threshold,
    }


def print_runtime_summary() -> None:
    print(f'PROJECT_ROOT: {PROJECT_ROOT}')
    print(f'DATA_ROOT   : {DATA_ROOT}')
    print(f'SAVE_DIR    : {SAVE_DIR}')
    print(f'DEVICE      : {DEVICE.type} | pin_memory={PIN_MEMORY} | use_amp={USE_AMP}')
    print(f'RESUME      : {RESUME_FROM_LAST} | EMA={USE_EMA} (decay={EMA_DECAY}) | folds={N_FOLDS}')
    print(f'MODELS      : {[spec["name"] for spec in MODEL_SPECS]}')
    print_path_status('LABELED_IMAGES_DIR', LABELED_IMAGES_DIR, IMAGE_EXTS, required=True)
    print_path_status('LABELED_MASKS_DIR', LABELED_MASKS_DIR, MASK_EXTS, required=True)
    print_path_status('UNLABELED_DIR', UNLABELED_DIR, IMAGE_EXTS, required=False)
    print_path_status('EXTERNAL_UNLABELED_DIR', EXTERNAL_UNLABELED_DIR, IMAGE_EXTS, required=False)
    print_path_status('TEST_IMAGES_DIR', TEST_IMAGES_DIR, IMAGE_EXTS, required=False)
    print_path_status('CLASSIFICATION_RECURSIVE_DIR', CLASSIFICATION_RECURSIVE_DIR, IMAGE_EXTS, required=False)


print_runtime_summary()

/Users/fgrach/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /Users/fgrach/University/MIET/labs/lab_object_segmentation
DATA_ROOT   : /Users/fgrach/University/MIET/labs/lab_object_segmentation/dl-lab-3-product-segmentation
SAVE_DIR    : /Users/fgrach/University/MIET/labs/lab_object_segmentation/seg_runs/advanced_baseline
DEVICE      : mps | pin_memory=False | use_amp=False
RESUME      : True | EMA=True (decay=0.999) | folds=3
MODELS      : ['unetpp_resnet34', 'fpn_resnet34', 'unet_resnet34']
[OK] LABELED_IMAGES_DIR: /Users/fgrach/University/MIET/labs/lab_object_segmentation/dl-lab-3-product-segmentation/train/images | files=2000
[OK] LABELED_MASKS_DIR: /Users/fgrach/University/MIET/labs/lab_object_segmentation/dl-lab-3-product-segmentation/train/masks | files=2000
[OK] UNLABELED_DIR: /Users/fgrach/University/MIET/labs/lab_object_segmentation/dl-lab-3-product-segmentation/unlabeled/images | files=350
[OPTIONAL] EXTERNAL_UNLABELED_DIR: /Users/fgrach/University/MIET/labs/lab_object_segmentation/external_unlabeled/images | missing -> w

In [3]:
from __future__ import annotations

# =========================
# DATASET / MODEL / TRAIN UTILS
# =========================

def collect_labeled_samples(images_dir: Path, masks_dir: Path) -> list[tuple[Path, Path]]:
    image_map = {
        image_path.stem: image_path
        for image_path in sorted(images_dir.rglob('*'))
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTS
    }

    samples: list[tuple[Path, Path]] = []
    missing_images: list[str] = []

    for mask_path in sorted(masks_dir.rglob('*')):
        if not mask_path.is_file() or mask_path.suffix.lower() not in MASK_EXTS:
            continue
        image_path = image_map.get(mask_path.stem)
        if image_path is None:
            missing_images.append(mask_path.name)
            continue
        samples.append((image_path, mask_path))

    if missing_images:
        print(f'[WARN] masks without matching images: {len(missing_images)}')
    if not samples:
        raise RuntimeError('No paired image/mask samples were found.')
    return samples


def sample_group_key(image_path: Path) -> str:
    parts = image_path.stem.split('_')
    return parts[0] if parts else image_path.stem


def collect_image_paths(input_dir: Path) -> list[Path]:
    return [
        path
        for path in sorted(input_dir.rglob('*'))
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS
    ]


def build_group_folds(
    samples: list[tuple[Path, Path]],
    n_folds: int,
    seed: int,
) -> list[dict]:
    grouped_samples: dict[str, list[tuple[Path, Path]]] = {}
    for image_path, mask_path in samples:
        group_key = sample_group_key(image_path)
        grouped_samples.setdefault(group_key, []).append((image_path, mask_path))

    group_items = list(grouped_samples.items())
    rng = np.random.default_rng(seed)
    rng.shuffle(group_items)
    group_items.sort(key=lambda item: len(item[1]), reverse=True)

    fold_buckets = [{'groups': [], 'samples': []} for _ in range(n_folds)]
    for group_key, group_samples in group_items:
        target_bucket = min(range(n_folds), key=lambda idx: len(fold_buckets[idx]['samples']))
        fold_buckets[target_bucket]['groups'].append(group_key)
        fold_buckets[target_bucket]['samples'].extend(group_samples)

    folds = []
    for fold_idx in range(n_folds):
        val_groups = set(fold_buckets[fold_idx]['groups'])
        val_samples = list(fold_buckets[fold_idx]['samples'])
        train_samples = []
        train_groups = []

        for other_idx in range(n_folds):
            if other_idx == fold_idx:
                continue
            train_samples.extend(fold_buckets[other_idx]['samples'])
            train_groups.extend(fold_buckets[other_idx]['groups'])

        folds.append(
            {
                'fold_idx': fold_idx,
                'train_samples': train_samples,
                'val_samples': val_samples,
                'train_group_count': len(train_groups),
                'val_group_count': len(val_groups),
                'train_sample_count': len(train_samples),
                'val_sample_count': len(val_samples),
                'train_groups_preview': sorted(train_groups)[:10],
                'val_groups_preview': sorted(val_groups)[:10],
            }
        )

    return folds


def build_train_transform(img_size: int) -> A.Compose:
    return A.Compose(
        [
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.10,
                rotate_limit=10,
                border_mode=cv2.BORDER_REFLECT_101,
                p=0.6,
            ),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
            A.GaussianBlur(blur_limit=(3, 5), p=0.15),
        ]
    )


def build_val_transform(img_size: int) -> A.Compose:
    return A.Compose([A.Resize(img_size, img_size)])


class BinarySegDataset(Dataset):
    def __init__(
        self,
        samples: list[tuple[Path, Path]],
        transform: A.Compose,
        encoder_name: str,
        encoder_weights: str | None,
    ):
        self.samples = samples
        self.transform = transform
        self.preprocess_input = None
        if encoder_weights is not None:
            self.preprocess_input = get_preprocessing_fn(encoder_name, pretrained=encoder_weights)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        image_path, mask_path = self.samples[idx]

        image_bgr = read_image(image_path, cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise RuntimeError(f'Cannot read image: {image_path}')
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        mask = read_image(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f'Cannot read mask: {mask_path}')
        mask = (mask > 0).astype(np.float32)

        transformed = self.transform(image=image_rgb, mask=mask)
        image_rgb = transformed['image'].astype(np.float32)
        mask = transformed['mask'].astype(np.float32)

        if self.preprocess_input is not None:
            image_rgb = self.preprocess_input(image_rgb)
        else:
            image_rgb = image_rgb / 255.0

        image = torch.from_numpy(image_rgb.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask[None, ...]).float()
        sample_id = image_path.name
        return image, mask, sample_id


def build_model(model_spec: dict | None = None) -> nn.Module:
    model_spec = DEFAULT_MODEL_SPEC if model_spec is None else model_spec
    model_name = model_spec['model_name']
    encoder_name = model_spec['encoder_name']
    encoder_weights = model_spec['encoder_weights']

    kwargs = {
        'encoder_name': encoder_name,
        'encoder_weights': encoder_weights,
        'in_channels': 3,
        'classes': NUM_CLASSES,
        'activation': ACTIVATION,
    }

    if model_name == 'UnetPlusPlus':
        return smp.UnetPlusPlus(**kwargs)
    if model_name == 'Unet':
        return smp.Unet(**kwargs)
    if model_name == 'FPN':
        return smp.FPN(**kwargs)
    raise ValueError(f'Unsupported model_name: {model_name}')


class CombinedBCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode=smp.losses.BINARY_MODE, from_logits=True)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = self.bce(logits, targets)
        dice = self.dice(logits, targets)
        return 0.5 * bce + 0.5 * dice


def dice_score(logits: torch.Tensor, targets: torch.Tensor, threshold: float = DEFAULT_THRESHOLD, eps: float = 1e-7) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    preds = preds.flatten(1)
    targets = targets.flatten(1)

    intersection = (preds * targets).sum(dim=1)
    denominator = preds.sum(dim=1) + targets.sum(dim=1)
    score = (2.0 * intersection + eps) / (denominator + eps)
    return score.mean().item()


def iou_score(logits: torch.Tensor, targets: torch.Tensor, threshold: float = DEFAULT_THRESHOLD, eps: float = 1e-7) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    preds = preds.flatten(1)
    targets = targets.flatten(1)

    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection
    score = (intersection + eps) / (union + eps)
    return score.mean().item()


def dice_score_np(probabilities: np.ndarray, targets: np.ndarray, threshold: float, eps: float = 1e-7) -> float:
    preds = (probabilities >= threshold).astype(np.float32).reshape(len(probabilities), -1)
    targets = targets.astype(np.float32).reshape(len(targets), -1)
    intersection = (preds * targets).sum(axis=1)
    denominator = preds.sum(axis=1) + targets.sum(axis=1)
    return float(((2.0 * intersection + eps) / (denominator + eps)).mean())


def iou_score_np(probabilities: np.ndarray, targets: np.ndarray, threshold: float, eps: float = 1e-7) -> float:
    preds = (probabilities >= threshold).astype(np.float32).reshape(len(probabilities), -1)
    targets = targets.astype(np.float32).reshape(len(targets), -1)
    intersection = (preds * targets).sum(axis=1)
    union = preds.sum(axis=1) + targets.sum(axis=1) - intersection
    return float(((intersection + eps) / (union + eps)).mean())


def autocast_context(device: torch.device, use_amp: bool):
    if not use_amp:
        return nullcontext()
    if device.type == 'mps':
        return torch.autocast(device_type='mps', dtype=torch.float16)
    if device.type == 'cuda':
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return nullcontext()


def build_grad_scaler(device: torch.device, use_amp: bool):
    if use_amp and device.type == 'cuda':
        return torch.cuda.amp.GradScaler()
    return None


def move_batch_to_device(batch, device: torch.device):
    images, masks, sample_ids = batch
    non_blocking = device.type == 'cuda'
    images = images.to(device, non_blocking=non_blocking)
    masks = masks.to(device, non_blocking=non_blocking)
    return images, masks, sample_ids


def move_optimizer_state(optimizer: torch.optim.Optimizer, device: torch.device) -> None:
    for state in optimizer.state.values():
        for key, value in state.items():
            if isinstance(value, torch.Tensor):
                state[key] = value.to(device)


def create_ema_model(model: nn.Module) -> nn.Module:
    ema_model = deepcopy(model)
    ema_model.eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    return ema_model


@torch.no_grad()
def update_ema(ema_model: nn.Module, model: nn.Module, decay: float) -> None:
    ema_state = ema_model.state_dict()
    model_state = model.state_dict()

    for key, ema_value in ema_state.items():
        model_value = model_state[key].detach()
        if torch.is_floating_point(ema_value):
            ema_value.mul_(decay).add_(model_value, alpha=1.0 - decay)
        else:
            ema_value.copy_(model_value)


def model_root_dir(model_spec: dict) -> Path:
    return SAVE_DIR / 'runs' / model_spec['name']


def fold_run_dir(model_spec: dict, fold_idx: int) -> Path:
    return model_root_dir(model_spec) / f'fold_{fold_idx}'


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
        drop_last=False,
    )


def save_history(run_dir: Path, history: list[dict], summary: dict | None = None) -> None:
    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    payload = {
        'summary': summary or {},
        'epochs': history,
    }
    save_json(run_dir / 'history.json', payload)


def build_run_config_payload(model_spec: dict, fold_idx: int, best_threshold: float | None = None) -> dict:
    payload = build_config_payload(best_threshold=best_threshold)
    payload['current_run'] = {
        'model_spec': model_spec,
        'fold_idx': fold_idx,
    }
    return payload


def checkpoint_payload(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.ReduceLROnPlateau,
    epoch: int,
    row: dict,
    config_payload: dict,
    best_threshold: float | None = None,
    ema_model: nn.Module | None = None,
    grad_scaler=None,
    training_state: dict | None = None,
) -> dict:
    return {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'ema_state_dict': ema_model.state_dict() if ema_model is not None else None,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'grad_scaler_state_dict': grad_scaler.state_dict() if grad_scaler is not None else None,
        'metrics': row,
        'config': config_payload,
        'training_state': training_state or {},
        'best_threshold': best_threshold,
    }


def load_training_state_from_last_checkpoint(
    model: nn.Module,
    ema_model: nn.Module | None,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.ReduceLROnPlateau,
    grad_scaler,
    run_dir: Path,
    config_payload: dict,
) -> dict:
    state = {
        'start_epoch': 1,
        'best_val_dice': -1.0,
        'best_epoch': 0,
        'best_threshold': DEFAULT_THRESHOLD,
        'epochs_without_improvement': 0,
        'history': [],
        'resumed': False,
    }

    checkpoint_path = run_dir / 'last_checkpoint.pth'
    if not RESUME_FROM_LAST or not checkpoint_path.exists():
        return state

    try:
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        checkpoint_run = checkpoint.get('config', {}).get('current_run', {})
        current_run = config_payload.get('current_run', {})
        checkpoint_signature = {
            'model_name': checkpoint_run.get('model_spec', {}).get('name'),
            'fold_idx': checkpoint_run.get('fold_idx'),
            'img_size': checkpoint.get('config', {}).get('train', {}).get('img_size'),
            'n_folds': checkpoint.get('config', {}).get('folds', {}).get('n_folds'),
            'use_ema': checkpoint.get('config', {}).get('train', {}).get('use_ema'),
        }
        current_signature = {
            'model_name': current_run.get('model_spec', {}).get('name'),
            'fold_idx': current_run.get('fold_idx'),
            'img_size': config_payload.get('train', {}).get('img_size'),
            'n_folds': config_payload.get('folds', {}).get('n_folds'),
            'use_ema': config_payload.get('train', {}).get('use_ema'),
        }
        if checkpoint_signature != current_signature:
            print(f'[WARN] Resume skipped for {run_dir.name}: config mismatch')
            return state

        model.load_state_dict(checkpoint['model_state_dict'])
        if ema_model is not None and checkpoint.get('ema_state_dict') is not None:
            ema_model.load_state_dict(checkpoint['ema_state_dict'])

        if checkpoint.get('optimizer_state_dict') is not None:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            move_optimizer_state(optimizer, DEVICE)
        if checkpoint.get('scheduler_state_dict') is not None:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        if grad_scaler is not None and checkpoint.get('grad_scaler_state_dict') is not None:
            grad_scaler.load_state_dict(checkpoint['grad_scaler_state_dict'])

        history_payload = load_json(run_dir / 'history.json', default={}) or {}
        training_state = checkpoint.get('training_state', {})
        best_threshold = checkpoint.get('best_threshold')
        if best_threshold is None:
            best_threshold = training_state.get('best_threshold', DEFAULT_THRESHOLD)

        state.update(
            {
                'start_epoch': int(checkpoint.get('epoch', 0)) + 1,
                'best_val_dice': float(training_state.get('best_val_dice', checkpoint.get('metrics', {}).get('val_dice', -1.0))),
                'best_epoch': int(training_state.get('best_epoch', checkpoint.get('epoch', 0))),
                'best_threshold': float(best_threshold),
                'epochs_without_improvement': int(training_state.get('epochs_without_improvement', 0)),
                'history': history_payload.get('epochs', []),
                'resumed': True,
            }
        )
        print(f'Resumed {current_signature["model_name"]} fold {current_signature["fold_idx"]} from epoch {state["start_epoch"]}')
        return state
    except Exception as exc:
        print(f'[WARN] Resume skipped for {run_dir.name}: {exc}')
        return state


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    device: torch.device,
    use_amp: bool,
    grad_scaler,
    ema_model: nn.Module | None = None,
    ema_decay: float = EMA_DECAY,
) -> dict:
    model.train()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    for batch in loader:
        images, masks, _ = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)

        with autocast_context(device, use_amp):
            logits = model(images)
            loss = loss_fn(logits, masks)

        if grad_scaler is not None:
            grad_scaler.scale(loss).backward()
            grad_scaler.step(optimizer)
            grad_scaler.update()
        else:
            loss.backward()
            optimizer.step()

        if ema_model is not None:
            update_ema(ema_model, model, ema_decay)

        running_loss += float(loss.item())
        running_dice += dice_score(logits.detach(), masks)
        running_iou += iou_score(logits.detach(), masks)

    num_batches = len(loader)
    return {
        'loss': running_loss / num_batches,
        'dice': running_dice / num_batches,
        'iou': running_iou / num_batches,
    }


@torch.no_grad()
def validate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: nn.Module,
    device: torch.device,
    use_amp: bool,
) -> dict:
    model.eval()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    for batch in loader:
        images, masks, _ = move_batch_to_device(batch, device)
        with autocast_context(device, use_amp):
            logits = model(images)
            loss = loss_fn(logits, masks)

        running_loss += float(loss.item())
        running_dice += dice_score(logits, masks)
        running_iou += iou_score(logits, masks)

    num_batches = len(loader)
    return {
        'loss': running_loss / num_batches,
        'dice': running_dice / num_batches,
        'iou': running_iou / num_batches,
    }


@torch.no_grad()
def collect_validation_outputs(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    use_amp: bool,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    model.eval()
    probabilities = []
    targets = []
    sample_ids: list[str] = []

    for batch in loader:
        images, masks, batch_sample_ids = move_batch_to_device(batch, device)
        with autocast_context(device, use_amp):
            logits = model(images)
        probabilities.append(torch.sigmoid(logits).cpu().numpy()[:, 0])
        targets.append(masks.cpu().numpy()[:, 0])
        sample_ids.extend(list(batch_sample_ids))

    return np.concatenate(probabilities, axis=0), np.concatenate(targets, axis=0), sample_ids


def tune_threshold(probabilities: np.ndarray, targets: np.ndarray) -> tuple[float, list[dict]]:
    rows: list[dict] = []
    best_threshold = DEFAULT_THRESHOLD
    best_dice = -1.0

    for threshold in THRESHOLD_GRID:
        dice = dice_score_np(probabilities, targets, threshold)
        iou = iou_score_np(probabilities, targets, threshold)
        row = {
            'threshold': float(threshold),
            'val_dice': float(dice),
            'val_iou': float(iou),
        }
        rows.append(row)
        if row['val_dice'] > best_dice:
            best_dice = row['val_dice']
            best_threshold = threshold

    return float(best_threshold), rows


def train_single_fold(model_spec: dict, fold_payload: dict) -> dict:
    fold_idx = int(fold_payload['fold_idx'])
    train_samples = fold_payload['train_samples']
    val_samples = fold_payload['val_samples']

    run_dir = fold_run_dir(model_spec, fold_idx)
    run_dir.mkdir(parents=True, exist_ok=True)
    save_json(
        run_dir / 'fold_summary.json',
        {
            'fold_idx': fold_idx,
            'model_name': model_spec['name'],
            **{key: value for key, value in fold_payload.items() if key not in {'train_samples', 'val_samples'}},
        },
    )

    train_dataset = BinarySegDataset(
        samples=train_samples,
        transform=build_train_transform(IMG_SIZE),
        encoder_name=model_spec['encoder_name'],
        encoder_weights=model_spec['encoder_weights'],
    )
    val_dataset = BinarySegDataset(
        samples=val_samples,
        transform=build_val_transform(IMG_SIZE),
        encoder_name=model_spec['encoder_name'],
        encoder_weights=model_spec['encoder_weights'],
    )

    train_loader = make_loader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = make_loader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = build_model(model_spec).to(DEVICE)
    ema_model = create_ema_model(model) if USE_EMA else None
    loss_fn = CombinedBCEDiceLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MIN_LR,
    )
    grad_scaler = build_grad_scaler(DEVICE, USE_AMP)

    config_payload = build_run_config_payload(model_spec=model_spec, fold_idx=fold_idx)
    resume_state = load_training_state_from_last_checkpoint(
        model=model,
        ema_model=ema_model,
        optimizer=optimizer,
        scheduler=scheduler,
        grad_scaler=grad_scaler,
        run_dir=run_dir,
        config_payload=config_payload,
    )

    effective_num_epochs = 1 if SMOKE_TEST_ONE_EPOCH else NUM_EPOCHS
    start_epoch = resume_state['start_epoch']
    best_val_dice = resume_state['best_val_dice']
    best_epoch = resume_state['best_epoch']
    best_threshold = resume_state['best_threshold']
    epochs_without_improvement = resume_state['epochs_without_improvement']
    history = resume_state['history']
    resumed = resume_state['resumed']

    training_started_at = time.perf_counter()

    for epoch in range(start_epoch, effective_num_epochs + 1):
        epoch_started_at = time.perf_counter()

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            use_amp=USE_AMP,
            grad_scaler=grad_scaler,
            ema_model=ema_model,
            ema_decay=EMA_DECAY,
        )

        eval_model = ema_model if ema_model is not None else model
        val_metrics = validate_one_epoch(
            model=eval_model,
            loader=val_loader,
            loss_fn=loss_fn,
            device=DEVICE,
            use_amp=USE_AMP,
        )

        scheduler.step(val_metrics['dice'])
        epoch_time_sec = time.perf_counter() - epoch_started_at

        row = {
            'epoch': epoch,
            'lr': float(optimizer.param_groups[0]['lr']),
            'train_loss': float(train_metrics['loss']),
            'train_dice': float(train_metrics['dice']),
            'train_iou': float(train_metrics['iou']),
            'val_loss': float(val_metrics['loss']),
            'val_dice': float(val_metrics['dice']),
            'val_iou': float(val_metrics['iou']),
            'epoch_time_sec': float(epoch_time_sec),
            'model_name': model_spec['name'],
            'fold_idx': fold_idx,
            'val_model': 'ema' if ema_model is not None else 'raw',
        }
        history.append(row)

        improved = row['val_dice'] > best_val_dice
        if improved:
            best_val_dice = row['val_dice']
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        training_state = {
            'best_val_dice': float(best_val_dice),
            'best_epoch': int(best_epoch),
            'best_threshold': float(best_threshold),
            'epochs_without_improvement': int(epochs_without_improvement),
            'history_length': len(history),
            'resumed': resumed,
        }

        torch.save(
            checkpoint_payload(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                row=row,
                config_payload=config_payload,
                best_threshold=best_threshold,
                ema_model=ema_model,
                grad_scaler=grad_scaler,
                training_state=training_state,
            ),
            run_dir / 'last_checkpoint.pth',
        )

        if improved:
            torch.save(
                checkpoint_payload(
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    epoch=epoch,
                    row=row,
                    config_payload=config_payload,
                    best_threshold=best_threshold,
                    ema_model=ema_model,
                    grad_scaler=grad_scaler,
                    training_state=training_state,
                ),
                run_dir / 'best_checkpoint.pth',
            )
            print(f'[{model_spec["name"]}][fold {fold_idx}] Saved new best checkpoint with val_dice={best_val_dice:.4f}')

        summary = {
            'best_val_dice': float(best_val_dice),
            'best_epoch': int(best_epoch),
            'best_threshold': float(best_threshold),
            'smoke_test_one_epoch': SMOKE_TEST_ONE_EPOCH,
            'resume_from_last': RESUME_FROM_LAST,
            'resumed': resumed,
            'use_ema': USE_EMA,
            'ema_decay': EMA_DECAY,
            'model_name': model_spec['name'],
            'fold_idx': fold_idx,
        }
        save_history(run_dir, history, summary=summary)

        print(
            f'[{model_spec["name"]}][fold {fold_idx}] '
            f'Epoch {epoch:02d}/{effective_num_epochs} | '
            f'lr={row["lr"]:.2e} | '
            f'train_loss={row["train_loss"]:.4f} train_dice={row["train_dice"]:.4f} train_iou={row["train_iou"]:.4f} | '
            f'val_loss={row["val_loss"]:.4f} val_dice={row["val_dice"]:.4f} val_iou={row["val_iou"]:.4f} | '
            f'eval_model={row["val_model"]} | '
            f'time={row["epoch_time_sec"] / 60.0:.1f} min'
        )

        if epoch == start_epoch:
            estimated_full_hours = (row['epoch_time_sec'] * NUM_EPOCHS) / 3600.0
            print(
                f'[{model_spec["name"]}][fold {fold_idx}] '
                f'Observed epoch time: {row["epoch_time_sec"] / 60.0:.1f} min. '
                f'Estimated {NUM_EPOCHS} epochs: {estimated_full_hours:.2f} h'
            )

        if not SMOKE_TEST_ONE_EPOCH and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f'[{model_spec["name"]}][fold {fold_idx}] Early stopping triggered after epoch {epoch}.')
            break

    training_time_sec = time.perf_counter() - training_started_at
    best_checkpoint_path = run_dir / 'best_checkpoint.pth'
    if not best_checkpoint_path.exists():
        raise FileNotFoundError(f'Best checkpoint was not created: {best_checkpoint_path}')

    best_checkpoint = torch.load(best_checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(best_checkpoint['model_state_dict'])
    model.to(DEVICE)
    if ema_model is not None and best_checkpoint.get('ema_state_dict') is not None:
        ema_model.load_state_dict(best_checkpoint['ema_state_dict'])
        ema_model.to(DEVICE)

    tuning_model = ema_model if ema_model is not None and best_checkpoint.get('ema_state_dict') is not None else model
    val_probabilities, val_targets, val_sample_ids = collect_validation_outputs(
        model=tuning_model,
        loader=val_loader,
        device=DEVICE,
        use_amp=USE_AMP,
    )
    best_threshold, threshold_rows = tune_threshold(val_probabilities, val_targets)

    np.save(run_dir / 'val_probs.npy', val_probabilities.astype(np.float16))
    np.save(run_dir / 'val_targets.npy', val_targets.astype(np.uint8))
    save_json(run_dir / 'val_sample_ids.json', {'sample_ids': val_sample_ids})

    threshold_payload = {
        'best_threshold': float(best_threshold),
        'threshold_grid': THRESHOLD_GRID,
        'rows': threshold_rows,
        'weights_source': 'ema' if best_checkpoint.get('ema_state_dict') is not None else 'raw',
    }
    save_json(run_dir / 'threshold_tuning.json', threshold_payload)

    for checkpoint_name in ['best_checkpoint.pth', 'last_checkpoint.pth']:
        checkpoint_path = run_dir / checkpoint_name
        if checkpoint_path.exists():
            checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
            checkpoint['best_threshold'] = float(best_threshold)
            checkpoint['config'] = build_run_config_payload(model_spec=model_spec, fold_idx=fold_idx, best_threshold=float(best_threshold))
            checkpoint.setdefault('training_state', {})
            checkpoint['training_state']['best_threshold'] = float(best_threshold)
            torch.save(checkpoint, checkpoint_path)

    summary = {
        'best_val_dice': float(best_val_dice),
        'best_epoch': int(best_epoch),
        'best_threshold': float(best_threshold),
        'training_time_sec': float(training_time_sec),
        'model_name': model_spec['name'],
        'fold_idx': fold_idx,
        'weights_source': threshold_payload['weights_source'],
    }
    save_json(run_dir / 'config.json', build_run_config_payload(model_spec=model_spec, fold_idx=fold_idx, best_threshold=float(best_threshold)))
    save_json(
        run_dir / 'history.json',
        {
            'summary': summary,
            'epochs': history,
            'threshold_tuning': threshold_rows,
        },
    )
    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)

    return {
        'model_name': model_spec['name'],
        'fold_idx': fold_idx,
        'run_dir': str(run_dir),
        'best_val_dice': float(best_val_dice),
        'best_epoch': int(best_epoch),
        'best_threshold': float(best_threshold),
        'weights_source': threshold_payload['weights_source'],
        'training_time_sec': float(training_time_sec),
    }


def model_fold_artifact_path(model_spec: dict, fold_idx: int, name: str) -> Path:
    return fold_run_dir(model_spec, fold_idx) / name


def evaluate_model_oof(model_spec: dict) -> dict:
    best_threshold = DEFAULT_THRESHOLD
    best_dice = -1.0
    best_iou = -1.0

    for threshold in THRESHOLD_GRID:
        total_dice = 0.0
        total_iou = 0.0
        total_samples = 0

        for fold_idx in range(N_FOLDS):
            probs = np.load(model_fold_artifact_path(model_spec, fold_idx, 'val_probs.npy'), mmap_mode='r')
            targets = np.load(model_fold_artifact_path(model_spec, fold_idx, 'val_targets.npy'), mmap_mode='r')
            fold_samples = int(targets.shape[0])
            total_dice += dice_score_np(np.asarray(probs), np.asarray(targets), threshold) * fold_samples
            total_iou += iou_score_np(np.asarray(probs), np.asarray(targets), threshold) * fold_samples
            total_samples += fold_samples

        mean_dice = total_dice / max(total_samples, 1)
        mean_iou = total_iou / max(total_samples, 1)
        if mean_dice > best_dice:
            best_dice = mean_dice
            best_iou = mean_iou
            best_threshold = threshold

    summary = {
        'model_name': model_spec['name'],
        'best_threshold': float(best_threshold),
        'oof_best_dice': float(best_dice),
        'oof_best_iou': float(best_iou),
    }
    save_json(model_root_dir(model_spec) / 'oof_summary.json', summary)
    return summary


def normalize_weight_map(weight_map: dict[str, float]) -> dict[str, float]:
    clipped = {name: max(0.0, float(weight)) for name, weight in weight_map.items()}
    total = sum(clipped.values())
    if total <= 0:
        equal = 1.0 / max(len(clipped), 1)
        return {name: equal for name in clipped}
    return {name: value / total for name, value in clipped.items()}


def fit_regression_ensemble_weights(model_specs: list[dict]) -> dict[str, float]:
    rng = np.random.default_rng(SEED)
    samples_per_fold = max(1, ENSEMBLE_PIXEL_SAMPLES // max(N_FOLDS, 1))
    x_chunks = []
    y_chunks = []

    for fold_idx in range(N_FOLDS):
        targets = np.load(model_fold_artifact_path(model_specs[0], fold_idx, 'val_targets.npy'), mmap_mode='r')
        target_flat = np.asarray(targets).reshape(-1).astype(np.float32)
        if len(target_flat) == 0:
            continue

        sample_size = min(samples_per_fold, len(target_flat))
        sampled_indices = rng.choice(len(target_flat), size=sample_size, replace=False)
        x_fold = []

        for model_spec in model_specs:
            probs = np.load(model_fold_artifact_path(model_spec, fold_idx, 'val_probs.npy'), mmap_mode='r')
            prob_flat = np.asarray(probs).reshape(-1).astype(np.float32)
            x_fold.append(prob_flat[sampled_indices])

        x_fold = np.stack(x_fold, axis=1)
        y_fold = target_flat[sampled_indices]
        x_chunks.append(x_fold)
        y_chunks.append(y_fold)

    x = np.concatenate(x_chunks, axis=0)
    y = np.concatenate(y_chunks, axis=0)
    weights = np.linalg.lstsq(x, y, rcond=None)[0]

    weight_map = {model_spec['name']: float(weight) for model_spec, weight in zip(model_specs, weights)}
    weight_map = normalize_weight_map(weight_map)
    weight_map = {
        name: (weight if weight >= ENSEMBLE_MIN_WEIGHT else 0.0)
        for name, weight in weight_map.items()
    }
    return normalize_weight_map(weight_map)


def evaluate_oof_candidate(weight_map: dict[str, float]) -> dict:
    active_weights = {name: weight for name, weight in weight_map.items() if weight > 0}
    if not active_weights:
        raise ValueError('No active models in candidate weight map.')

    best_threshold = DEFAULT_THRESHOLD
    best_dice = -1.0
    best_iou = -1.0

    for threshold in THRESHOLD_GRID:
        total_dice = 0.0
        total_iou = 0.0
        total_samples = 0

        for fold_idx in range(N_FOLDS):
            targets = np.load(model_fold_artifact_path(DEFAULT_MODEL_SPEC, fold_idx, 'val_targets.npy'), mmap_mode='r')
            combined = np.zeros(targets.shape, dtype=np.float32)

            for model_spec in MODEL_SPECS:
                model_name = model_spec['name']
                if model_name not in active_weights:
                    continue
                probs = np.load(model_fold_artifact_path(model_spec, fold_idx, 'val_probs.npy'), mmap_mode='r')
                combined += float(active_weights[model_name]) * np.asarray(probs, dtype=np.float32)

            fold_samples = int(targets.shape[0])
            total_dice += dice_score_np(combined, np.asarray(targets), threshold) * fold_samples
            total_iou += iou_score_np(combined, np.asarray(targets), threshold) * fold_samples
            total_samples += fold_samples

        mean_dice = total_dice / max(total_samples, 1)
        mean_iou = total_iou / max(total_samples, 1)
        if mean_dice > best_dice:
            best_dice = mean_dice
            best_iou = mean_iou
            best_threshold = threshold

    return {
        'weights_by_model': active_weights,
        'best_threshold': float(best_threshold),
        'oof_best_dice': float(best_dice),
        'oof_best_iou': float(best_iou),
        'selected_models': sorted(active_weights),
    }


def build_best_ensemble(model_summaries: list[dict]) -> dict:
    ranked_models = sorted(model_summaries, key=lambda row: row['oof_best_dice'], reverse=True)
    best_single = ranked_models[0]

    equal_all_weights = normalize_weight_map({spec['name']: 1.0 for spec in MODEL_SPECS})
    regression_weights = fit_regression_ensemble_weights(MODEL_SPECS)
    top2_models = ranked_models[:2]
    top2_equal_weights = normalize_weight_map({row['model_name']: 1.0 for row in top2_models})
    best_single_weights = {best_single['model_name']: 1.0}

    candidates = [
        {'name': 'best_single', 'weights_by_model': best_single_weights},
        {'name': 'equal_all_models', 'weights_by_model': equal_all_weights},
        {'name': 'top2_equal', 'weights_by_model': top2_equal_weights},
        {'name': 'regression_weights', 'weights_by_model': regression_weights},
    ]

    evaluated_candidates = []
    for candidate in candidates:
        candidate_result = evaluate_oof_candidate(candidate['weights_by_model'])
        candidate_result['candidate_name'] = candidate['name']
        evaluated_candidates.append(candidate_result)

    best_candidate = dict(max(evaluated_candidates, key=lambda row: row['oof_best_dice']))
    candidate_rankings = [dict(row) for row in evaluated_candidates]
    best_candidate['candidate_rankings'] = candidate_rankings
    save_json(SAVE_DIR / 'ensemble_candidates.json', {'candidates': candidate_rankings})
    save_json(SAVE_DIR / 'ensemble_summary.json', best_candidate)
    return best_candidate


In [4]:
# =========================
# NIGHT TRAINING: 3 FOLDS x 2-3 MODELS
# =========================
seed_everything(SEED)

print()
print('[Path check before training]')
print_path_status('LABELED_IMAGES_DIR', LABELED_IMAGES_DIR, IMAGE_EXTS, required=True)
print_path_status('LABELED_MASKS_DIR', LABELED_MASKS_DIR, MASK_EXTS, required=True)

if not LABELED_IMAGES_DIR.exists():
    raise FileNotFoundError(f'Labeled images directory does not exist: {LABELED_IMAGES_DIR}')
if not LABELED_MASKS_DIR.exists():
    raise FileNotFoundError(f'Labeled masks directory does not exist: {LABELED_MASKS_DIR}')

samples = collect_labeled_samples(LABELED_IMAGES_DIR, LABELED_MASKS_DIR)
print(f'Paired labeled samples: {len(samples)}')

folds = build_group_folds(samples=samples, n_folds=N_FOLDS, seed=SEED)
save_json(
    SAVE_DIR / 'folds_summary.json',
    {
        'n_folds': N_FOLDS,
        'folds': [
            {
                key: value
                for key, value in fold_payload.items()
                if key not in {'train_samples', 'val_samples'}
            }
            for fold_payload in folds
        ],
    },
)

print('Fold summary:')
for fold_payload in folds:
    print(
        f'  fold={fold_payload["fold_idx"]} | '
        f'train_samples={fold_payload["train_sample_count"]} | '
        f'val_samples={fold_payload["val_sample_count"]} | '
        f'train_groups={fold_payload["train_group_count"]} | '
        f'val_groups={fold_payload["val_group_count"]}'
    )

training_results = []
model_summaries = []

for model_spec in MODEL_SPECS:
    print()
    print(f'===== Training model: {model_spec["name"]} =====')

    for fold_payload in folds:
        result = train_single_fold(model_spec=model_spec, fold_payload=fold_payload)
        training_results.append(result)

    model_oof_summary = evaluate_model_oof(model_spec)
    model_summaries.append(model_oof_summary)
    print(
        f'[OOF] {model_spec["name"]} | '
        f'best_threshold={model_oof_summary["best_threshold"]:.2f} | '
        f'best_dice={model_oof_summary["oof_best_dice"]:.4f} | '
        f'best_iou={model_oof_summary["oof_best_iou"]:.4f}'
    )

save_json(SAVE_DIR / 'training_results.json', {'runs': training_results, 'model_summaries': model_summaries})
pd.DataFrame(training_results).to_csv(SAVE_DIR / 'training_results.csv', index=False)
pd.DataFrame(model_summaries).to_csv(SAVE_DIR / 'model_summaries.csv', index=False)

ensemble_summary = build_best_ensemble(model_summaries)
print()
print('===== Best ensemble =====')
print('candidate_name :', ensemble_summary['candidate_name'])
print('selected_models:', ensemble_summary['selected_models'])
print('weights_by_model:', ensemble_summary['weights_by_model'])
print('best_threshold :', ensemble_summary['best_threshold'])
print('oof_best_dice  :', ensemble_summary['oof_best_dice'])
print('oof_best_iou   :', ensemble_summary['oof_best_iou'])
print('Saved ensemble summary to:', SAVE_DIR / 'ensemble_summary.json')


[Path check before training]
[OK] LABELED_IMAGES_DIR: /Users/fgrach/University/MIET/labs/lab_object_segmentation/dl-lab-3-product-segmentation/train/images | files=2000
[OK] LABELED_MASKS_DIR: /Users/fgrach/University/MIET/labs/lab_object_segmentation/dl-lab-3-product-segmentation/train/masks | files=2000
Paired labeled samples: 2000
Fold summary:
  fold=0 | train_samples=1183 | val_samples=817 | train_groups=14 | val_groups=1
  fold=1 | train_samples=1406 | val_samples=594 | train_groups=9 | val_groups=6
  fold=2 | train_samples=1411 | val_samples=589 | train_groups=7 | val_groups=8

===== Training model: unetpp_resnet34 =====


/Users/fgrach/Library/Python/3.9/lib/python/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Resumed unetpp_resnet34 fold 0 from epoch 36
[unetpp_resnet34][fold 0] Epoch 36/60 | lr=1.87e-05 | train_loss=0.0347 train_dice=0.9361 train_iou=0.8913 | val_loss=0.1219 val_dice=0.8526 val_iou=0.7724 | eval_model=ema | time=1.2 min
[unetpp_resnet34][fold 0] Observed epoch time: 1.2 min. Estimated 60 epochs: 1.19 h
[unetpp_resnet34][fold 0] Early stopping triggered after epoch 36.
Resumed unetpp_resnet34 fold 1 from epoch 24
[unetpp_resnet34][fold 1] Epoch 24/60 | lr=3.75e-05 | train_loss=0.0393 train_dice=0.9332 train_iou=0.8865 | val_loss=0.0953 val_dice=0.8499 val_iou=0.7654 | eval_model=ema | time=1.3 min
[unetpp_resnet34][fold 1] Observed epoch time: 1.3 min. Estimated 60 epochs: 1.31 h
[unetpp_resnet34][fold 1] Early stopping triggered after epoch 24.
Resumed unetpp_resnet34 fold 2 from epoch 38
[unetpp_resnet34][fold 2] Epoch 38/60 | lr=3.75e-05 | train_loss=0.0307 train_dice=0.9551 train_iou=0.9187 | val_loss=0.1677 val_dice=0.7067 val_iou=0.6257 | eval_model=ema | time=1.4 min

## Inference Helpers

Loads `best_checkpoint.pth`, applies TTA with original + horizontal flip, postprocesses masks, and saves both `.png` masks and optional `.npy` probability maps.


In [5]:
from __future__ import annotations

# =========================
# ENSEMBLE INFERENCE / PSEUDO LABELING UTILS
# =========================

def load_run_model(run_dir: Path):
    checkpoint_path = run_dir / 'best_checkpoint.pth'
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint does not exist: {checkpoint_path}')

    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    checkpoint_config = checkpoint.get('config', {})
    current_run = checkpoint_config.get('current_run', {})
    model_spec = current_run.get('model_spec', DEFAULT_MODEL_SPEC)

    encoder_name = model_spec.get('encoder_name', DEFAULT_MODEL_SPEC['encoder_name'])
    encoder_weights = model_spec.get('encoder_weights', DEFAULT_MODEL_SPEC['encoder_weights'])
    img_size = checkpoint_config.get('train', {}).get('img_size', IMG_SIZE)
    best_threshold = checkpoint.get('best_threshold')
    if best_threshold is None:
        best_threshold = checkpoint_config.get('best_threshold', DEFAULT_THRESHOLD)

    inference_spec = dict(model_spec)
    inference_spec['encoder_weights'] = None
    model = build_model(inference_spec)

    state_dict_key = 'ema_state_dict' if checkpoint.get('ema_state_dict') is not None else 'model_state_dict'
    model.load_state_dict(checkpoint[state_dict_key])
    model.to(DEVICE)
    model.eval()

    preprocess_input = None
    if encoder_weights is not None:
        preprocess_input = get_preprocessing_fn(encoder_name, pretrained=encoder_weights)

    return model, preprocess_input, int(img_size), float(best_threshold), model_spec


def preprocess_rgb_image(image_rgb: np.ndarray, img_size: int, preprocess_input):
    image_resized = cv2.resize(image_rgb, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image_resized = image_resized.astype(np.float32)

    if preprocess_input is not None:
        image_resized = preprocess_input(image_resized)
    else:
        image_resized = image_resized / 255.0

    tensor = torch.from_numpy(image_resized.transpose(2, 0, 1)).float().unsqueeze(0)
    return tensor.to(DEVICE)


@torch.no_grad()
def predict_single_model_probability_map(
    model: nn.Module,
    image_rgb: np.ndarray,
    img_size: int,
    preprocess_input,
    use_amp: bool,
    use_tta: bool = True,
) -> np.ndarray:
    height, width = image_rgb.shape[:2]

    inputs = [image_rgb]
    if use_tta:
        inputs.append(image_rgb[:, ::-1].copy())

    probability_maps = []
    for index, image_variant in enumerate(inputs):
        tensor = preprocess_rgb_image(image_variant, img_size, preprocess_input)
        with autocast_context(DEVICE, use_amp):
            logits = model(tensor)
        probs = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        if index == 1:
            probs = probs[:, ::-1]
        probability_maps.append(probs)

    mean_probs = np.mean(probability_maps, axis=0)
    if mean_probs.shape != (height, width):
        mean_probs = cv2.resize(mean_probs.astype(np.float32), (width, height), interpolation=cv2.INTER_LINEAR)
    return mean_probs.astype(np.float32)


def remove_small_components(mask: np.ndarray, min_area: int) -> np.ndarray:
    mask_uint8 = mask.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_uint8, connectivity=8)
    cleaned = np.zeros_like(mask_uint8)

    for label_idx in range(1, num_labels):
        area = int(stats[label_idx, cv2.CC_STAT_AREA])
        if area >= min_area:
            cleaned[labels == label_idx] = 1

    return cleaned


def postprocess_mask(mask: np.ndarray) -> np.ndarray:
    min_area = max(16, int(mask.size * POSTPROCESS_MIN_COMPONENT_AREA_RATIO))
    cleaned = remove_small_components(mask, min_area=min_area)

    if POSTPROCESS_CLOSE_KERNEL > 1:
        kernel = np.ones((POSTPROCESS_CLOSE_KERNEL, POSTPROCESS_CLOSE_KERNEL), dtype=np.uint8)
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)
        cleaned = (cleaned > 0).astype(np.uint8)

    return cleaned


def load_ensemble_members(ensemble_summary_path: Path):
    ensemble_summary = load_json(ensemble_summary_path)
    if not ensemble_summary:
        raise FileNotFoundError(f'Ensemble summary does not exist: {ensemble_summary_path}')

    members = []
    for model_name, model_weight in ensemble_summary['weights_by_model'].items():
        model_spec = next(spec for spec in MODEL_SPECS if spec['name'] == model_name)

        available_folds = []
        for fold_idx in range(N_FOLDS):
            run_dir = fold_run_dir(model_spec, fold_idx)
            if (run_dir / 'best_checkpoint.pth').exists():
                available_folds.append((fold_idx, run_dir))

        if not available_folds:
            continue

        per_fold_weight = float(model_weight) / len(available_folds)
        for fold_idx, run_dir in available_folds:
            model, preprocess_input, img_size, _, _ = load_run_model(run_dir)
            members.append(
                {
                    'model_name': model_name,
                    'fold_idx': fold_idx,
                    'weight': per_fold_weight,
                    'model': model,
                    'preprocess_input': preprocess_input,
                    'img_size': img_size,
                }
            )

    return ensemble_summary, members


@torch.no_grad()
def predict_ensemble_probability_map(
    members: list[dict],
    image_rgb: np.ndarray,
    use_amp: bool,
    use_tta: bool = True,
) -> np.ndarray:
    height, width = image_rgb.shape[:2]
    combined = np.zeros((height, width), dtype=np.float32)

    for member in members:
        probs = predict_single_model_probability_map(
            model=member['model'],
            image_rgb=image_rgb,
            img_size=member['img_size'],
            preprocess_input=member['preprocess_input'],
            use_amp=use_amp,
            use_tta=use_tta,
        )
        combined += float(member['weight']) * probs

    return combined.astype(np.float32)


def relative_output_paths(input_root: Path, output_root: Path, image_path: Path) -> tuple[Path, Path]:
    relative_path = image_path.relative_to(input_root)
    mask_path = (output_root / 'masks' / relative_path).with_suffix('.png')
    prob_path = (output_root / 'probs' / relative_path).with_suffix('.npy')
    return mask_path, prob_path


def run_ensemble_inference_on_directory(
    ensemble_summary_path: Path,
    input_dir: Path,
    output_dir: Path,
    threshold: float | None = None,
) -> list[dict]:
    if not input_dir.exists():
        raise FileNotFoundError(f'Input directory does not exist: {input_dir}')

    image_paths = collect_image_paths(input_dir)
    if not image_paths:
        raise FileNotFoundError(f'No images found in: {input_dir}')

    ensemble_summary, members = load_ensemble_members(ensemble_summary_path)
    threshold = ensemble_summary['best_threshold'] if threshold is None else float(threshold)

    rows = []
    print(f'Running ensemble inference for {len(image_paths)} images from {input_dir}')
    print(f'Ensemble members: {len(members)}')
    print(f'Threshold       : {threshold:.2f}')

    for index, image_path in enumerate(image_paths, 1):
        image_bgr = read_image(image_path, cv2.IMREAD_COLOR)
        if image_bgr is None:
            print(f'[skip] cannot read image: {image_path}')
            continue

        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        probabilities = predict_ensemble_probability_map(
            members=members,
            image_rgb=image_rgb,
            use_amp=USE_AMP,
            use_tta=True,
        )
        mask = postprocess_mask((probabilities >= threshold).astype(np.uint8))

        mask_path, prob_path = relative_output_paths(input_root=input_dir, output_root=output_dir, image_path=image_path)
        save_png_mask(mask_path, mask.astype(np.uint8) * 255)
        if SAVE_PROBABILITY_MAPS:
            prob_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(prob_path, probabilities.astype(np.float16))

        rows.append(
            {
                'image_name': image_path.name,
                'relative_path': str(image_path.relative_to(input_dir)),
                'image_path': str(image_path),
                'mask_path': str(mask_path),
                'prob_path': str(prob_path) if SAVE_PROBABILITY_MAPS else None,
                'area_ratio': float(mask.mean()),
                'mean_probability': float(probabilities.mean()),
            }
        )

        if index % 100 == 0 or index == len(image_paths):
            print(f'Processed {index}/{len(image_paths)}')

    pd.DataFrame(rows).to_csv(output_dir / 'stats.csv', index=False)
    save_json(
        output_dir / 'summary.json',
        {
            'input_dir': str(input_dir),
            'output_dir': str(output_dir),
            'num_images': len(rows),
            'threshold': threshold,
            'ensemble_summary_path': str(ensemble_summary_path),
            'selected_models': ensemble_summary['selected_models'],
            'weights_by_model': ensemble_summary['weights_by_model'],
        },
    )
    return rows


def pseudo_sample_stats(probabilities: np.ndarray, mask: np.ndarray) -> dict:
    confidence_map = np.maximum(probabilities, 1.0 - probabilities)
    foreground_probs = probabilities[mask > 0]
    return {
        'mean_confidence': float(confidence_map.mean()),
        'min_confidence': float(confidence_map.min()),
        'foreground_mean_probability': float(foreground_probs.mean()) if foreground_probs.size else 0.0,
        'area_ratio': float(mask.mean()),
    }


def is_confident_pseudo_sample(stats: dict) -> tuple[bool, list[str]]:
    reasons: list[str] = []

    if stats['mean_confidence'] < PSEUDO_MIN_MEAN_CONFIDENCE:
        reasons.append('low_mean_confidence')
    if PSEUDO_MIN_PIXEL_CONFIDENCE is not None and stats['min_confidence'] < PSEUDO_MIN_PIXEL_CONFIDENCE:
        reasons.append('low_min_confidence')
    if stats['area_ratio'] < PSEUDO_MIN_AREA_RATIO:
        reasons.append('mask_too_small')
    if stats['area_ratio'] > PSEUDO_MAX_AREA_RATIO:
        reasons.append('mask_too_large')
    if stats['area_ratio'] > 0 and stats['foreground_mean_probability'] < PSEUDO_MIN_FOREGROUND_PROB:
        reasons.append('weak_foreground_probability')

    return len(reasons) == 0, reasons


def generate_ensemble_pseudo_labels_for_directory(
    ensemble_summary_path: Path,
    input_dir: Path,
    dataset_name: str,
    output_dir: Path,
    threshold: float | None = None,
    save_only_confident: bool = PSEUDO_SAVE_ONLY_CONFIDENT,
) -> list[dict]:
    if not input_dir.exists():
        print(f'[skip] {dataset_name}: directory does not exist -> {input_dir}')
        return []

    image_paths = collect_image_paths(input_dir)
    if not image_paths:
        print(f'[skip] {dataset_name}: no images found in {input_dir}')
        return []

    ensemble_summary, members = load_ensemble_members(ensemble_summary_path)
    threshold = ensemble_summary['best_threshold'] if threshold is None else float(threshold)

    rows = []
    saved_count = 0

    print(f'Generating ensemble pseudo labels for {dataset_name}: {len(image_paths)} images')
    print(f'Input dir       : {input_dir}')
    print(f'Output dir      : {output_dir}')
    print(f'Threshold       : {threshold:.2f}')
    print(f'Ensemble members: {len(members)}')

    for index, image_path in enumerate(image_paths, 1):
        image_bgr = read_image(image_path, cv2.IMREAD_COLOR)
        if image_bgr is None:
            print(f'[skip] cannot read image: {image_path}')
            continue

        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        probabilities = predict_ensemble_probability_map(
            members=members,
            image_rgb=image_rgb,
            use_amp=USE_AMP,
            use_tta=True,
        )
        mask = postprocess_mask((probabilities >= threshold).astype(np.uint8))
        stats = pseudo_sample_stats(probabilities, mask)
        is_confident, reasons = is_confident_pseudo_sample(stats)
        should_save = is_confident or not save_only_confident

        mask_path, prob_path = relative_output_paths(input_root=input_dir, output_root=output_dir, image_path=image_path)
        if should_save:
            save_png_mask(mask_path, mask.astype(np.uint8) * 255)
            if SAVE_PROBABILITY_MAPS:
                prob_path.parent.mkdir(parents=True, exist_ok=True)
                np.save(prob_path, probabilities.astype(np.float16))
            saved_count += 1

        row = {
            'image_name': image_path.name,
            'relative_path': str(image_path.relative_to(input_dir)),
            'image_path': str(image_path),
            'saved': bool(should_save),
            'is_confident': bool(is_confident),
            'reasons': ';'.join(reasons),
            'mask_path': str(mask_path) if should_save else None,
            'prob_path': str(prob_path) if should_save and SAVE_PROBABILITY_MAPS else None,
            'threshold': float(threshold),
            **stats,
        }
        rows.append(row)

        if index % 100 == 0 or index == len(image_paths):
            print(f'Processed {index}/{len(image_paths)}')

    pd.DataFrame(rows).to_csv(output_dir / 'stats.csv', index=False)
    save_json(output_dir / 'stats.json', {'rows': rows})
    save_json(
        output_dir / 'summary.json',
        {
            'dataset_name': dataset_name,
            'input_dir': str(input_dir),
            'output_dir': str(output_dir),
            'num_images': len(rows),
            'saved_count': saved_count,
            'save_only_confident': save_only_confident,
            'threshold': threshold,
            'selected_models': ensemble_summary['selected_models'],
            'weights_by_model': ensemble_summary['weights_by_model'],
        },
    )
    print(f'{dataset_name}: saved {saved_count}/{len(rows)} pseudo labels')
    return rows

In [6]:
# =========================
# ENSEMBLE TEST INFERENCE
# =========================
ENSEMBLE_SUMMARY_PATH = SAVE_DIR / 'ensemble_summary.json'
TEST_OUTPUT_DIR = SAVE_DIR / 'ensemble_outputs' / 'test_images'

if RUN_TEST_INFERENCE:
    test_rows = run_ensemble_inference_on_directory(
        ensemble_summary_path=ENSEMBLE_SUMMARY_PATH,
        input_dir=TEST_IMAGES_DIR,
        output_dir=TEST_OUTPUT_DIR,
    )
    print(f'Ensemble test inference saved to: {TEST_OUTPUT_DIR}')
    print(pd.DataFrame(test_rows).head())
else:
    print('RUN_TEST_INFERENCE = False -> skipping ensemble test inference.')

RUN_TEST_INFERENCE = False -> skipping ensemble test inference.


In [7]:
# =========================
# ENSEMBLE PSEUDO LABELING
# =========================
ENSEMBLE_SUMMARY_PATH = SAVE_DIR / 'ensemble_summary.json'

if RUN_PSEUDO_LABELING:
    unlabeled_rows = generate_ensemble_pseudo_labels_for_directory(
        ensemble_summary_path=ENSEMBLE_SUMMARY_PATH,
        input_dir=UNLABELED_DIR,
        dataset_name='unlabeled',
        output_dir=SAVE_DIR / 'ensemble_outputs' / 'unlabeled',
        save_only_confident=False,
    )
    external_rows = generate_ensemble_pseudo_labels_for_directory(
        ensemble_summary_path=ENSEMBLE_SUMMARY_PATH,
        input_dir=EXTERNAL_UNLABELED_DIR,
        dataset_name='external_unlabeled',
        output_dir=SAVE_DIR / 'ensemble_outputs' / 'external_unlabeled',
        save_only_confident=False,
    )
    print(f"Ensemble pseudo-label root: {SAVE_DIR / 'ensemble_outputs'}")
    if unlabeled_rows:
        print(pd.DataFrame(unlabeled_rows).head())
else:
    print('RUN_PSEUDO_LABELING = False -> skipping ensemble pseudo labeling.')

if RUN_CLASSIFICATION_RECURSIVE_PSEUDO:
    classification_rows = generate_ensemble_pseudo_labels_for_directory(
        ensemble_summary_path=ENSEMBLE_SUMMARY_PATH,
        input_dir=CLASSIFICATION_RECURSIVE_DIR,
        dataset_name='dl_lab_1_image_classification_all',
        output_dir=SAVE_DIR / 'ensemble_outputs' / 'dl_lab_1_image_classification_all',
        save_only_confident=False,
    )
    if classification_rows:
        print(pd.DataFrame(classification_rows).head())
else:
    print('RUN_CLASSIFICATION_RECURSIVE_PSEUDO = False -> skipping classification recursive pseudo labeling.')

Generating ensemble pseudo labels for unlabeled: 350 images
Input dir       : /Users/fgrach/University/MIET/labs/lab_object_segmentation/dl-lab-3-product-segmentation/unlabeled/images
Output dir      : /Users/fgrach/University/MIET/labs/lab_object_segmentation/seg_runs/advanced_baseline/ensemble_outputs/unlabeled
Threshold       : 0.40
Ensemble members: 9
Processed 100/350
Processed 200/350
Processed 300/350
Processed 350/350
unlabeled: saved 350/350 pseudo labels
[skip] external_unlabeled: directory does not exist -> /Users/fgrach/University/MIET/labs/lab_object_segmentation/external_unlabeled/images
Ensemble pseudo-label root: /Users/fgrach/University/MIET/labs/lab_object_segmentation/seg_runs/advanced_baseline/ensemble_outputs
                                          image_name  \
0  10.107.215.111_20260119194016_ae644d4b-b36b-40...   
1  10.107.215.111_20260119203341_bb74ac11-3f14-4a...   
2  10.107.224.111_20260118140224_dfc3593c-6938-42...   
3  10.107.224.111_20260118141134_d43